# 02 – Molecular Features

Convert SMILES into Morgan/ECFP4 fingerprints and interpretable RDKit descriptors. Fingerprints capture local substructure patterns while descriptors provide chemically interpretable context for toxicity modeling.

In [2]:
from pathlib import Path
import sys, json
import pandas as pd

ROOT = Path.cwd().resolve().parent
if not (ROOT/"data").exists(): ROOT = Path.cwd().resolve()
sys.path.insert(0, str(ROOT))

from src.preprocessing import load_tox21, TOX21_ENDPOINTS
from src.molecular_features import featurize_smiles

path = ROOT/"data/raw/tox21/tox21.csv"
if not path.exists(): path = ROOT/"tox21.csv"
df = load_tox21(path)
X, descriptors, invalid = featurize_smiles(df["smiles"], radius=2, n_bits=2048)

Y = df[TOX21_ENDPOINTS].apply(pd.to_numeric, errors="coerce")
metadata = df[["mol_id","smiles"]].copy()

print("X:", X.shape, "Y:", Y.shape)
print("Invalid SMILES:", int(invalid.sum()))
display(descriptors.describe().T)

[01:24:12] WARNING: not removing hydrogen atom without neighbors
[01:24:12] Explicit valence for atom # 8 Al, 6, is greater than permitted
[01:24:13] Explicit valence for atom # 3 Al, 6, is greater than permitted
[01:24:13] Explicit valence for atom # 4 Al, 6, is greater than permitted
[01:24:13] Explicit valence for atom # 4 Al, 6, is greater than permitted
[01:24:14] Explicit valence for atom # 9 Al, 6, is greater than permitted
[01:24:14] Explicit valence for atom # 5 Al, 6, is greater than permitted
[01:24:14] Explicit valence for atom # 16 Al, 6, is greater than permitted
[01:24:15] Explicit valence for atom # 20 Al, 6, is greater than permitted
[01:24:25] DEPRECATION WARNING: please use MorganGenerator
[01:24:25] DEPRECATION WARNING: please use MorganGenerator
[01:24:25] DEPRECATION WARNING: please use MorganGenerator
[01:24:25] DEPRECATION WARNING: please use MorganGenerator
[01:24:25] DEPRECATION WARNING: please use MorganGenerator
[01:24:25] DEPRECATION WARNING: please use Mor

X: (7831, 2057) Y: (7831, 12)
Invalid SMILES: 8


,count,mean,std,min,25%,50%,75%,max
MolWt,7823.0,276.144155,164.732356,9.0120,165.236000,240.302000,343.044000,1877.6640
MolLogP,7823.0,2.373943,2.304307,-17.4064,1.149350,2.365500,3.653150,22.6118
TPSA,7823.0,59.472201,57.794990,0.0000,26.300000,46.530000,77.030000,777.9800
HBA,7823.0,3.473987,3.115448,0.0000,2.000000,3.000000,4.000000,46.0000
HBD,7823.0,1.224978,1.916035,0.0000,0.000000,1.000000,2.000000,30.0000
RotatableBonds,7823.0,4.302697,4.464772,0.0000,1.000000,3.000000,6.000000,47.0000
RingCount,7823.0,1.772721,1.667745,0.0000,1.000000,1.000000,3.000000,30.0000
HeavyAtomCount,7823.0,18.566918,11.309542,1.0000,11.000000,16.000000,23.000000,132.0000
FractionCSP3,7823.0,0.458506,0.325090,0.0000,0.181818,0.416667,0.726136,1.0000


In [3]:
OUT = ROOT/"data/processed/tox21"
OUT.mkdir(parents=True, exist_ok=True)

X.to_parquet(OUT/"X_tox21.parquet")
Y.to_parquet(OUT/"Y_tox21.parquet")
metadata.to_parquet(OUT/"tox21_metadata.parquet")
pd.DataFrame({"mol_id": df.loc[invalid,"mol_id"], "smiles": df.loc[invalid,"smiles"]}).to_csv(
    OUT/"invalid_smiles_log.csv", index=False
)

with open(OUT/"feature_config.json","w") as f:
    json.dump({"radius":2,"n_bits":2048,"descriptor_columns":list(descriptors.columns),
               "endpoints":TOX21_ENDPOINTS}, f, indent=2)
print("Features saved.")

Features saved.


### Why these features?

Morgan fingerprints are a strong baseline for structure–activity learning. RDKit descriptors expose physicochemical signals such as lipophilicity, polarity, hydrogen bonding, and molecular size that can be communicated to medicinal-chemistry and toxicology teams.

**TODO:** add organization-specific structure standardization, salt/tautomer policy, stereochemistry policy, and duplicate handling.